# บทที่ 6: การพัฒนาโครงข่ายประสาทเทียม

ใน Notebook นี้ เราจะเรียนรู้กระบวนการพัฒนาโครงข่ายประสาทเทียม ตั้งแต่การเตรียมข้อมูล การเลือกสถาปัตยกรรม การฝึกฝน และการปรับแต่งไฮเพอร์พารามิเตอร์ (Hyperparameter Tuning)

**ศัพท์ที่สำคัญในบทนี้:**- เวกเตอร์ (vector) — อาร์เรย์หนึ่งมิติ- เมทริกซ์ (matrix) — อาร์เรย์สองมิติ- ค่าน้ำหนัก (weight) — พารามิเตอร์ที่ปรับได้- ค่าไบแอส (bias) — ค่าเลื่อน- อินพุต (input) — ข้อมูลนำเข้า- เอาต์พุต (output) — ผลลัพธ์- ค่าสูญเสีย (loss) — วัดความคลาดเคลื่อน- เกรเดียนต์ (gradient) — ทิศทางการปรับ- ฟังก์ชันกระตุ้น (activation function) — ฟังก์ชันไม่เป็นเชิงเส้น- โครงข่ายประสาทเทียม (neural network) — โมเดลแมชชีนเลิร์นนิง

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
try:
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'],
                   capture_output=True)
except FileNotFoundError:
    pass  # เครื่องที่ไม่มี apt-get (macOS/Windows) ใช้ฟอนต์ไทยที่ติดตั้งไว้ในเครื่องแทน

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score

np.random.seed(42)

## 2. การเตรียมข้อมูล (Data Preparation)

In [ ]:
# โหลดข้อมูล
data = load_iris()
X, y = data.data, data.target

# แบ่งข้อมูล
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"ชุดข้อมูล: Iris")
print(f"คุณลักษณะ: {X.shape[1]}")
print(f"ตัวอย่าง: {X.shape[0]}")
print(f"คลาส: {len(np.unique(y))}")

### 2.1 การจัดการค่าที่หายไปและค่าผิดปกติ (Missing Values & Outliers)

In [ ]:
# สาธิตการเติมค่าที่หายไปด้วยค่าเฉลี่ยและมัธยฐาน
data_with_nan = np.array([12.0, 15.0, np.nan, 18.0, 21.0, np.nan, 24.0])

mean_value = np.nanmean(data_with_nan)
median_value = np.nanmedian(data_with_nan)

filled_mean = np.where(np.isnan(data_with_nan), mean_value, data_with_nan)
filled_median = np.where(np.isnan(data_with_nan), median_value, data_with_nan)

print("=== การเติมค่าที่หายไป ===")
print(f"ข้อมูลเดิม: {data_with_nan}")
print(f"เติมด้วยค่าเฉลี่ย ({mean_value:.2f}): {filled_mean}")
print(f"เติมด้วยมัธยฐาน ({median_value:.2f}): {filled_median}")


def quartiles_by_median_split(sorted_data):
    """คำนวณ Q1 และ Q3 ด้วยวิธีแบ่งข้อมูลเป็นสองครึ่งแล้วหามัธยฐานของแต่ละครึ่ง (ตามตัวอย่างในบท)"""
    n = len(sorted_data)
    mid = n // 2
    lower_half = sorted_data[:mid]
    upper_half = sorted_data[mid:] if n % 2 == 0 else sorted_data[mid + 1:]
    return np.median(lower_half), np.median(upper_half)


def detect_outliers_iqr(data):
    """ตรวจจับค่าผิดปกติด้วยวิธีพิสัยระหว่างควอร์ไทล์ (IQR)"""
    sorted_data = np.sort(data)
    q1, q3 = quartiles_by_median_split(sorted_data)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    return q1, q3, iqr, lower_bound, upper_bound, outliers


def detect_outliers_zscore(data, threshold=3):
    """ตรวจจับค่าผิดปกติด้วยวิธีคะแนนมาตรฐาน"""
    mu = np.mean(data)
    sigma = np.std(data)
    z_scores = (data - mu) / sigma
    outliers = data[np.abs(z_scores) > threshold]
    return mu, sigma, z_scores, outliers


# ทวนตัวอย่างการตรวจจับค่าผิดปกติด้วยวิธี IQR ในบท (ข้อมูลรายได้ หน่วยหมื่นบาท)
income = np.array([15, 20, 25, 30, 35, 40, 45, 50, 55, 200])
q1, q3, iqr, lb, ub, outliers_iqr = detect_outliers_iqr(income)
print("\n=== วิธี IQR (ข้อมูลรายได้ หน่วยหมื่นบาท) ===")
print(f"ข้อมูล: {income}")
print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"ขอบเขต: [{lb}, {ub}]")
print(f"ค่าผิดปกติที่พบ: {outliers_iqr}")

# ทวนตัวอย่างการตรวจจับค่าผิดปกติด้วยวิธีคะแนนมาตรฐานในบท (ข้อมูลอุณหภูมิ)
temps = np.array([20, 22, 24, 26, 28, 30, 32, 34, 36, 50])
mu, sigma, z_scores, outliers_z = detect_outliers_zscore(temps)
print("\n=== วิธีคะแนนมาตรฐาน (ข้อมูลอุณหภูมิ) ===")
print(f"ข้อมูล: {temps}")
print(f"mu={mu:.2f}, sigma={sigma:.2f}")
print(f"คะแนนซี: {np.round(z_scores, 2)}")
if len(outliers_z):
    print(f"ค่าผิดปกติที่พบ (|z|>3): {outliers_z}")
else:
    print("ค่าผิดปกติที่พบ (|z|>3): ไม่พบ (แม้ 50 จะมีคะแนนซี ~2.41 ก็ยังไม่เกินเกณฑ์ ต่างจากผลของวิธี IQR)")

### 2.2 การปรับข้อมูลให้เป็นมาตรฐาน (Normalization)

In [ ]:
# StandardScaler (การทำมาตรฐานแบบคะแนนซี)
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)

# MinMaxScaler (ปรับสเกลต่ำสุด-สูงสุด 0-1)
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

print("=== ข้อมูลเดิม (คุณลักษณะที่ 0) ===")
print(f"ค่าเฉลี่ย: {X_train[:, 0].mean():.4f}, ส่วนเบี่ยงเบนมาตรฐาน: {X_train[:, 0].std():.4f}")
print(f"ต่ำสุด: {X_train[:, 0].min():.4f}, สูงสุด: {X_train[:, 0].max():.4f}")

print("\n=== StandardScaler (คุณลักษณะที่ 0) ===")
print(f"ค่าเฉลี่ย: {X_train_std[:, 0].mean():.4f}, ส่วนเบี่ยงเบนมาตรฐาน: {X_train_std[:, 0].std():.4f}")

print("\n=== MinMaxScaler (คุณลักษณะที่ 0) ===")
print(f"ต่ำสุด: {X_train_minmax[:, 0].min():.4f}, สูงสุด: {X_train_minmax[:, 0].max():.4f}")

## 3. การเลือกสถาปัตยกรรม

In [ ]:
def count_parameters(layer_sizes):
    """คำนวณจำนวนพารามิเตอร์"""
    total = 0
    for i in range(len(layer_sizes) - 1):
        weights = layer_sizes[i] * layer_sizes[i+1]
        biases = layer_sizes[i+1]
        total += weights + biases
    return total

# เปรียบเทียบสถาปัตยกรรมต่างๆ
architectures = [
    [4, 8, 3],           # เล็ก
    [4, 16, 8, 3],       # กลาง
    [4, 32, 16, 8, 3],   # ใหญ่
]

print("=== เปรียบเทียบสถาปัตยกรรม ===")
for arch in architectures:
    params = count_parameters(arch)
    print(f"{arch}: {params} พารามิเตอร์")

### 3.1 ฟังก์ชันกระตุ้น (Activation Functions)

นอกจากจำนวนพารามิเตอร์แล้ว การเลือกสถาปัตยกรรมยังต้องกำหนดฟังก์ชันกระตุ้นของแต่ละชั้น ด้านล่างสาธิต ReLU สำหรับชั้นซ่อน ซิกมอยด์สำหรับการจำแนกสองคลาส และ Softmax สำหรับการจำแนกหลายคลาส (รวมขั้นตอนลบลอจิตค่าสูงสุดก่อนยกกำลังเพื่อความเสถียรเชิงตัวเลข) พร้อมตรวจสอบผลลัพธ์กับตัวอย่างคำนวณในบท

In [ ]:
def relu(x):
    """ฟังก์ชันกระตุ้น ReLU: ReLU(x) = max(0, x)"""
    return np.maximum(0, x)

def sigmoid(x):
    """ฟังก์ชันกระตุ้นซิกมอยด์: sigma(x) = 1 / (1 + e^(-x))"""
    return 1 / (1 + np.exp(-x))

def softmax(z):
    """ฟังก์ชันกระตุ้น Softmax แบบเสถียรเชิงตัวเลข
    ลบลอจิตค่าสูงสุดออกก่อนยกกำลังเพื่อป้องกัน overflow (ผลลัพธ์เท่าเดิม)"""
    z_shifted = z - np.max(z)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z)


# ทวนตัวอย่างการคำนวณ ReLU ในบท
print("=== ฟังก์ชันกระตุ้น ReLU (ชั้นซ่อน) ===")
for val in [5, -3, 0, 2.5, -10]:
    print(f"ReLU({val}) = {relu(val)}")

# ทวนตัวอย่างการคำนวณซิกมอยด์ในบท (z = 2.5 -> ~0.924)
print("\n=== ฟังก์ชันกระตุ้นซิกมอยด์ (การจำแนกสองคลาส) ===")
z = 2.5
sig_z = sigmoid(z)
print(f"sigma({z}) = {sig_z:.3f}  (คาดหวังจากบท ~0.924)")
for val in [0, -1, 3]:
    print(f"sigma({val}) = {sigmoid(val):.3f}")

assert np.isclose(sig_z, 0.924, atol=1e-3), "ค่าซิกมอยด์ไม่ตรงกับตัวอย่างในบท"

# ทวนตัวอย่างการคำนวณ Softmax ในบท (z = [2.0, 1.0, 0.1] -> ~[0.659, 0.242, 0.099])
print("\n=== ฟังก์ชันกระตุ้น Softmax (การจำแนกหลายคลาส) ===")
z_logits = np.array([2.0, 1.0, 0.1])
softmax_result = softmax(z_logits)
print(f"Softmax({z_logits}) = {np.round(softmax_result, 3)}  (คาดหวังจากบท ~[0.659, 0.242, 0.099])")
print(f"ผลรวม = {softmax_result.sum():.4f}")

assert np.allclose(softmax_result, [0.659, 0.242, 0.099], atol=1e-3), "ค่า Softmax ไม่ตรงกับตัวอย่างในบท"

# สาธิตความเสถียรเชิงตัวเลข: ลบลอจิตค่าสูงสุดก่อนยกกำลังให้ผลลัพธ์เหมือนเดิม
z_max = np.max(z_logits)
z_prime = z_logits - z_max
print(f"\nลอจิตหลังลบค่าสูงสุด (z_max={z_max}): {z_prime}")
print(f"Softmax จากลอจิตที่ปรับแล้ว = {np.round(softmax(z_prime), 3)} (ผลลัพธ์เหมือนเดิมแต่เสถียรกว่าเชิงตัวเลข)")

## 4. ตารางอัตราการเรียนรู้ (Learning Rate Schedules)

In [ ]:
def constant_lr(epoch, initial_lr):
    """อัตราการเรียนรู้คงที่"""
    return initial_lr

def step_decay(epoch, initial_lr, drop_rate=0.5, epochs_drop=10):
    """การลดแบบขั้นบันได"""
    return initial_lr * (drop_rate ** (epoch // epochs_drop))

def exponential_decay(epoch, initial_lr, decay_rate=0.95):
    """การลดแบบเลขชี้กำลัง"""
    return initial_lr * (decay_rate ** epoch)

def cosine_annealing(epoch, initial_lr, total_epochs):
    """การลดทอนแบบโคไซน์"""
    return initial_lr * 0.5 * (1 + np.cos(np.pi * epoch / total_epochs))

# แสดงภาพ
epochs = 100
initial_lr = 0.1

x = range(epochs)
y_constant = [constant_lr(e, initial_lr) for e in x]
y_step = [step_decay(e, initial_lr) for e in x]
y_exp = [exponential_decay(e, initial_lr) for e in x]
y_cosine = [cosine_annealing(e, initial_lr, epochs) for e in x]

plt.figure(figsize=(10, 6))
plt.plot(x, y_constant, label='คงที่')
plt.plot(x, y_step, label='ขั้นบันได')
plt.plot(x, y_exp, label='เลขชี้กำลัง')
plt.plot(x, y_cosine, label='โคไซน์')
plt.xlabel('รอบการฝึก')
plt.ylabel('อัตราการเรียนรู้')
plt.title('ตารางอัตราการเรียนรู้')
plt.legend()
plt.show()

## 5. เส้นโค้งการเรียนรู้ (Learning Curves)

In [ ]:
def plot_learning_curves(train_losses, val_losses, title):
    """แสดงเส้นโค้งการเรียนรู้"""
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='ค่าสูญเสียชุดฝึก')
    plt.plot(val_losses, label='ค่าสูญเสียชุดตรวจสอบ')
    plt.xlabel('รอบการฝึก')
    plt.ylabel('ค่าสูญเสีย')
    plt.title(title)
    plt.legend()
    plt.show()

# จำลองเส้นโค้งการเรียนรู้ (ข้อมูลจำลองเพื่อประกอบการอธิบายรูปแบบเส้นโค้ง ไม่ใช่ผลจากการฝึกโมเดลจริง)
epochs = 100
train_loss_good = 1 / (1 + np.exp(-0.05 * (np.arange(epochs) - 20))) * 0.3 + 0.1
val_loss_good = train_loss_good + 0.05

train_loss_overfit = 1 / (1 + np.exp(-0.1 * (np.arange(epochs) - 10))) * 0.3 + 0.05
val_loss_overfit = np.concatenate([train_loss_overfit[:50] + 0.05,
                                   train_loss_overfit[50:] + 0.1 + np.linspace(0, 0.3, 50)])

print("=== ความพอดี (Good Fit) ===")
plot_learning_curves(train_loss_good, val_loss_good, 'ความพอดี')

print("=== การเรียนรู้เกินพอดี (Overfitting) ===")
plot_learning_curves(train_loss_overfit, val_loss_overfit, 'การเรียนรู้เกินพอดี')

## 6. การปรับแต่งไฮเพอร์พารามิเตอร์ (Hyperparameter Tuning)

In [ ]:
# ตัวอย่างการค้นหาแบบตาราง (Grid Search)
def grid_search_demo():
    """สาธิตแนวคิดการค้นหาแบบตาราง (ผลลัพธ์เป็นค่าจำลอง ไม่ได้ฝึกโมเดลจริง)"""
    learning_rates = [0.001, 0.01, 0.1, 0.5]
    hidden_sizes = [8, 16, 32, 64]

    print("=== พื้นที่การค้นหาแบบตาราง ===")
    print(f"อัตราการเรียนรู้: {learning_rates}")
    print(f"ขนาดชั้นซ่อน: {hidden_sizes}")
    print(f"จำนวนชุดค่าผสมทั้งหมด: {len(learning_rates) * len(hidden_sizes)}")

    # จำลองผลลัพธ์
    print("\nผลลัพธ์จำลอง:")
    best_acc = 0
    best_params = None

    for lr in learning_rates:
        for hs in hidden_sizes:
            # จำลองค่าความถูกต้อง
            acc = 0.7 + 0.2 * np.exp(-((lr - 0.1)**2) / 0.01) * (1 - np.exp(-hs/20))
            acc += np.random.uniform(-0.02, 0.02)
            print(f"lr={lr}, hidden={hs}: ความถูกต้อง={acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_params = (lr, hs)

    print(f"\nดีที่สุด: lr={best_params[0]}, hidden={best_params[1]}, ความถูกต้อง={best_acc:.4f}")

grid_search_demo()

## 7. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: คำนวณพารามิเตอร์

In [ ]:
# จงคำนวณจำนวนพารามิเตอร์ของ MLP ที่มีโครงสร้าง [784, 256, 128, 10]

layer_sizes = [784, 256, 128, 10]
total = count_parameters(layer_sizes)

print(f"สถาปัตยกรรม: {layer_sizes}")
print(f"พารามิเตอร์ทั้งหมด: {total:,}")

### แบบฝึกหัดที่ 2: การลดอัตราการเรียนรู้ (Learning Rate Decay)

In [ ]:
# ให้ initial_lr = 0.1, decay_rate = 0.9
# จงคำนวณอัตราการเรียนรู้ที่รอบการฝึก 0, 5, 10, 20

initial_lr = 0.1
decay_rate = 0.9
epochs_to_check = [0, 5, 10, 20]

for epoch in epochs_to_check:
    lr = initial_lr * (decay_rate ** epoch)
    print(f"รอบการฝึก {epoch}: อัตราการเรียนรู้ = {lr:.6f}")

### แบบฝึกหัดที่ 3: ผลของขนาดแบตช์ (Batch Size)

In [ ]:
# ให้ชุดข้อมูลมี 1000 ตัวอย่าง
# จงคำนวณจำนวนรอบอัปเดตต่อรอบการฝึก สำหรับขนาดแบตช์ = 16, 32, 64, 128

n_samples = 1000
batch_sizes = [16, 32, 64, 128]

print(f"ขนาดชุดข้อมูล: {n_samples}")
print("\nจำนวนรอบอัปเดตต่อรอบการฝึก:")
for bs in batch_sizes:
    iterations = n_samples // bs
    print(f"ขนาดแบตช์ {bs}: {iterations} รอบอัปเดต")

### แบบฝึกหัดที่ 4: การปรับข้อมูลให้เป็นมาตรฐาน (Normalization)

In [ ]:
# ให้ข้อมูล x = [10, 20, 30, 40, 50]
# จงปรับข้อมูลให้เป็นมาตรฐานด้วยคะแนนซีและ MinMax

x = np.array([10, 20, 30, 40, 50])

# การทำมาตรฐานแบบคะแนนซี
x_zscore = (x - x.mean()) / x.std()

# การปรับสเกลต่ำสุด-สูงสุด
x_minmax = (x - x.min()) / (x.max() - x.min())

print(f"ข้อมูลเดิม: {x}")
print(f"คะแนนซี: {x_zscore}")
print(f"MinMax: {x_minmax}")

## บทสรุป

Notebook นี้ครอบคลุม:
1. **การเตรียมข้อมูล**: การจัดการค่าที่หายไปและค่าผิดปกติ การปรับข้อมูลให้เป็นมาตรฐาน การแบ่งชุดข้อมูล
2. **การเลือกสถาปัตยกรรม**: การนับพารามิเตอร์ และฟังก์ชันกระตุ้น (ReLU, ซิกมอยด์, Softmax)
3. **ตารางอัตราการเรียนรู้**: คงที่ ขั้นบันได เลขชี้กำลัง โคไซน์
4. **เส้นโค้งการเรียนรู้**: การวินิจฉัยความเอนเอียง/ความแปรปรวน
5. **การปรับแต่งไฮเพอร์พารามิเตอร์**: การค้นหาแบบตาราง